In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [3]:
path = '/content/drive/MyDrive/sales_project/'

accounts = pd.read_csv(path + 'accounts.csv')
products = pd.read_csv(path + 'products.csv')
sales_pipeline = pd.read_csv(path + 'sales_pipeline.csv')
sales_teams = pd.read_csv(path + 'sales_teams.csv')
data_dict = pd.read_csv(path + 'data_dictionary.csv')

print("All files loaded successfully!")

All files loaded successfully!


In [4]:
tables = {
    'sales_pipeline': sales_pipeline,
    'accounts': accounts,
    'products': products,
    'sales_teams': sales_teams
}

for name, df in tables.items():
    print("=" * 50)
    print(f"TABLE: {name}")
    print(f"Rows: {df.shape[0]}  |  Columns: {df.shape[1]}")
    print("-" * 50)
    print("Column names:", df.columns.tolist())
    print("-" * 50)
    print(df.head(3))
    print()

TABLE: sales_pipeline
Rows: 8800  |  Columns: 8
--------------------------------------------------
Column names: ['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
--------------------------------------------------
  opportunity_id      sales_agent         product  account deal_stage  \
0       1C1I7A6R      Moses Frase  GTX Plus Basic  Cancity        Won   
1       Z063OYW0  Darcel Schlecht          GTXPro    Isdom        Won   
2       EC4QE1BX  Darcel Schlecht      MG Special  Cancity        Won   

  engage_date  close_date  close_value  
0  2016-10-20  2017-03-01       1054.0  
1  2016-10-25  2017-03-11       4514.0  
2  2016-10-25  2017-03-07         50.0  

TABLE: accounts
Rows: 85  |  Columns: 7
--------------------------------------------------
Column names: ['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
--------------------------------------------------
      

In [5]:
# Convert date columns to proper datetime format first
sales_pipeline['engage_date'] = pd.to_datetime(sales_pipeline['engage_date'])
sales_pipeline['close_date'] = pd.to_datetime(sales_pipeline['close_date'])

# Check all unique deal stages
print("DEAL STAGES & COUNT")
print("=" * 40)
print(sales_pipeline['deal_stage'].value_counts())

print("\n")

# Check date ranges
print("DATE RANGE")
print("=" * 40)
print("Engage date min:", sales_pipeline['engage_date'].min())
print("Engage date max:", sales_pipeline['engage_date'].max())
print("Close date min:", sales_pipeline['close_date'].min())
print("Close date max:", sales_pipeline['close_date'].max())

print("\n")

# Check close_value basics
print("CLOSE VALUE STATS")
print("=" * 40)
print(sales_pipeline['close_value'].describe())

print("\n")

# Check nulls in every column
print("NULL VALUES PER COLUMN")
print("=" * 40)
print(sales_pipeline.isnull().sum())

DEAL STAGES & COUNT
deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64


DATE RANGE
Engage date min: 2016-10-20 00:00:00
Engage date max: 2017-12-27 00:00:00
Close date min: 2017-03-01 00:00:00
Close date max: 2017-12-31 00:00:00


CLOSE VALUE STATS
count     6711.000000
mean      1490.915512
std       2320.670773
min          0.000000
25%          0.000000
50%        472.000000
75%       3225.000000
max      30288.000000
Name: close_value, dtype: float64


NULL VALUES PER COLUMN
opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64


In [6]:
# Total deals that ENTERED the pipeline
total_deals = len(sales_pipeline)

# Deals at each stage
prospecting = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Prospecting'])
engaging = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Engaging'])
won = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Won'])
lost = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Lost'])

# Closed deals = Won + Lost
closed = won + lost

print("SALES FUNNEL OVERVIEW")
print("=" * 40)
print(f"Total Deals in Pipeline  : {total_deals}")
print(f"Prospecting              : {prospecting}")
print(f"Engaging                 : {engaging}")
print(f"Won                      : {won}")
print(f"Lost                     : {lost}")
print(f"Total Closed             : {closed}")

print("\n")

# Conversion rates between stages
prospect_to_engage = round((engaging + won + lost) / total_deals * 100, 1)
engage_to_close = round(closed / (engaging + closed) * 100, 1)
close_to_win = round(won / closed * 100, 1)
overall = round(won / total_deals * 100, 1)

print("CONVERSION RATES")
print("=" * 40)
print(f"Prospecting → Engaging   : {prospect_to_engage}%")
print(f"Engaging → Closed        : {engage_to_close}%")
print(f"Closed → Won             : {close_to_win}%")
print(f"Overall Win Rate         : {overall}%")

SALES FUNNEL OVERVIEW
Total Deals in Pipeline  : 8800
Prospecting              : 500
Engaging                 : 1589
Won                      : 4238
Lost                     : 2473
Total Closed             : 6711


CONVERSION RATES
Prospecting → Engaging   : 94.3%
Engaging → Closed        : 80.9%
Closed → Won             : 63.2%
Overall Win Rate         : 48.2%


In [7]:
# Revenue actually earned
total_revenue_won = sales_pipeline[sales_pipeline['deal_stage'] == 'Won']['close_value'].sum()

# Revenue lost in Lost deals
total_revenue_lost = sales_pipeline[sales_pipeline['deal_stage'] == 'Lost']['close_value'].sum()

# Won deals with ZERO value - hidden leakage
zero_value_won = sales_pipeline[
    (sales_pipeline['deal_stage'] == 'Won') &
    (sales_pipeline['close_value'] == 0)
]
zero_value_leakage = len(zero_value_won)

# Average deal value
avg_deal_value = sales_pipeline[
    sales_pipeline['deal_stage'] == 'Won'
]['close_value'].mean()

# Potential revenue sitting in active pipeline
active_engaging = sales_pipeline[sales_pipeline['deal_stage'] == 'Engaging']
potential_revenue = len(active_engaging) * avg_deal_value

print("REVENUE ANALYSIS")
print("=" * 40)
print(f"Total Revenue Earned (Won)     : ${total_revenue_won:,.2f}")
print(f"Revenue Lost (Lost deals)      : ${total_revenue_lost:,.2f}")
print(f"Won deals with ZERO value      : {zero_value_leakage} deals")
print(f"Average Deal Value             : ${avg_deal_value:,.2f}")
print(f"Potential Revenue in Pipeline  : ${potential_revenue:,.2f}")

print("\n")

# Leakage from zero value won deals
hidden_leakage = zero_value_leakage * avg_deal_value
print("REVENUE LEAKAGE SUMMARY")
print("=" * 40)
print(f"Hidden leakage (zero value Won): ${hidden_leakage:,.2f}")
print(f"Lost stage leakage             : ${total_revenue_lost:,.2f}")
print(f"Total Recoverable Revenue      : ${(hidden_leakage + total_revenue_lost):,.2f}")

REVENUE ANALYSIS
Total Revenue Earned (Won)     : $10,005,534.00
Revenue Lost (Lost deals)      : $0.00
Won deals with ZERO value      : 0 deals
Average Deal Value             : $2,360.91
Potential Revenue in Pipeline  : $3,751,485.02


REVENUE LEAKAGE SUMMARY
Hidden leakage (zero value Won): $0.00
Lost stage leakage             : $0.00
Total Recoverable Revenue      : $0.00


In [8]:
# Average deal value from Won deals only
avg_deal_value = sales_pipeline[
    sales_pipeline['deal_stage'] == 'Won'
]['close_value'].mean()

# Lost deals count
total_lost_deals = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Lost'])

# Prospecting deals that never moved forward
total_prospecting = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Prospecting'])

# Engaging deals still stuck
total_engaging_active = len(sales_pipeline[sales_pipeline['deal_stage'] == 'Engaging'])

# Revenue leakage from Lost deals (estimated)
lost_deal_leakage = total_lost_deals * avg_deal_value

# Revenue at risk in stalled pipeline
prospecting_at_risk = total_prospecting * avg_deal_value
engaging_at_risk = total_engaging_active * avg_deal_value

print("CORRECTED REVENUE LEAKAGE ANALYSIS")
print("=" * 45)
print(f"Average Won Deal Value          : ${avg_deal_value:,.2f}")
print(f"Total Won Revenue               : $10,005,534.00")

print("\n")

print("LEAKAGE BREAKDOWN")
print("=" * 45)
print(f"Lost Deals                      : {total_lost_deals} deals")
print(f"Estimated Lost Revenue          : ${lost_deal_leakage:,.2f}")

print("\n")

print("STALLED PIPELINE (AT RISK)")
print("=" * 45)
print(f"Stuck in Prospecting            : {total_prospecting} deals")
print(f"Revenue at Risk (Prospecting)   : ${prospecting_at_risk:,.2f}")
print(f"Stuck in Engaging               : {total_engaging_active} deals")
print(f"Revenue at Risk (Engaging)      : ${engaging_at_risk:,.2f}")

print("\n")

total_leakage = lost_deal_leakage + prospecting_at_risk + engaging_at_risk
print("=" * 45)
print(f"TOTAL RECOVERABLE REVENUE       : ${total_leakage:,.2f}")
print(f"TOTAL WON REVENUE               : $10,005,534.00")
print(f"LEAKAGE AS % OF WON REVENUE     : {round(total_leakage/10005534*100, 1)}%")

CORRECTED REVENUE LEAKAGE ANALYSIS
Average Won Deal Value          : $2,360.91
Total Won Revenue               : $10,005,534.00


LEAKAGE BREAKDOWN
Lost Deals                      : 2473 deals
Estimated Lost Revenue          : $5,838,528.92


STALLED PIPELINE (AT RISK)
Stuck in Prospecting            : 500 deals
Revenue at Risk (Prospecting)   : $1,180,454.70
Stuck in Engaging               : 1589 deals
Revenue at Risk (Engaging)      : $3,751,485.02


TOTAL RECOVERABLE REVENUE       : $10,770,468.64
TOTAL WON REVENUE               : $10,005,534.00
LEAKAGE AS % OF WON REVENUE     : 107.6%


In [9]:
avg_deal_value = 2360.91
total_lost_deals = 2473
total_prospecting = 500
total_engaging_active = 1589

# Realistic recovery assumption (25% of lost deals are recoverable)
recovery_rate = 0.25

# Realistic recoverable revenue
recoverable_from_lost = total_lost_deals * avg_deal_value * recovery_rate
recoverable_from_prospecting = total_prospecting * avg_deal_value * recovery_rate
recoverable_from_engaging = total_engaging_active * avg_deal_value * recovery_rate

total_realistic_recovery = (recoverable_from_lost +
                            recoverable_from_prospecting +
                            recoverable_from_engaging)

print("REALISTIC REVENUE LEAKAGE SUMMARY")
print("=" * 45)
print(f"Total Won Revenue               : $10,005,534.00")

print("\n")

print("LEAKAGE SOURCES")
print("=" * 45)
print(f"Lost Deals ({total_lost_deals} deals)")
print(f"  → Full estimated loss         : ${total_lost_deals * avg_deal_value:,.2f}")
print(f"  → Realistically recoverable   : ${recoverable_from_lost:,.2f}")

print("\n")

print(f"Stalled in Prospecting ({total_prospecting} deals)")
print(f"  → Revenue at risk             : ${total_prospecting * avg_deal_value:,.2f}")
print(f"  → Realistically recoverable   : ${recoverable_from_prospecting:,.2f}")

print("\n")

print(f"Stalled in Engaging ({total_engaging_active} deals)")
print(f"  → Revenue at risk             : ${total_engaging_active * avg_deal_value:,.2f}")
print(f"  → Realistically recoverable   : ${recoverable_from_engaging:,.2f}")

print("\n")

print("=" * 45)
print(f"TOTAL REALISTIC RECOVERY        : ${total_realistic_recovery:,.2f}")
print(f"LEAKAGE AS % OF WON REVENUE     : {round(total_realistic_recovery/10005534*100, 1)}%")

print("\n")

print("PORTFOLIO HEADLINE")
print("=" * 45)
print(f"Identified 3 pipeline stages with significant drop-off")
print(f"representing ${total_realistic_recovery:,.2f} in recoverable revenue")
print(f"against a total won revenue base of $10,005,534.00")

REALISTIC REVENUE LEAKAGE SUMMARY
Total Won Revenue               : $10,005,534.00


LEAKAGE SOURCES
Lost Deals (2473 deals)
  → Full estimated loss         : $5,838,530.43
  → Realistically recoverable   : $1,459,632.61


Stalled in Prospecting (500 deals)
  → Revenue at risk             : $1,180,455.00
  → Realistically recoverable   : $295,113.75


Stalled in Engaging (1589 deals)
  → Revenue at risk             : $3,751,485.99
  → Realistically recoverable   : $937,871.50


TOTAL REALISTIC RECOVERY        : $2,692,617.85
LEAKAGE AS % OF WON REVENUE     : 26.9%


PORTFOLIO HEADLINE
Identified 3 pipeline stages with significant drop-off
representing $2,692,617.85 in recoverable revenue
against a total won revenue base of $10,005,534.00


In [10]:
# Merge sales_pipeline with sales_teams to get regional data
pipeline_with_region = sales_pipeline.merge(sales_teams, on='sales_agent', how='left')

# Filter only Lost deals
lost_deals = pipeline_with_region[pipeline_with_region['deal_stage'] == 'Lost']

print("LOST DEALS BY PRODUCT")
print("=" * 45)
product_loss = lost_deals['product'].value_counts()
product_loss_pct = round(lost_deals['product'].value_counts(normalize=True) * 100, 1)
product_summary = pd.DataFrame({
    'Lost Deals': product_loss,
    'Percentage': product_loss_pct
})
print(product_summary)

print("\n")

print("LOST DEALS BY REGION")
print("=" * 45)
region_loss = lost_deals['regional_office'].value_counts()
region_loss_pct = round(lost_deals['regional_office'].value_counts(normalize=True) * 100, 1)
region_summary = pd.DataFrame({
    'Lost Deals': region_loss,
    'Percentage': region_loss_pct
})
print(region_summary)

print("\n")

print("LOST DEALS BY MANAGER")
print("=" * 45)
manager_loss = lost_deals['manager'].value_counts()
manager_loss_pct = round(lost_deals['manager'].value_counts(normalize=True) * 100, 1)
manager_summary = pd.DataFrame({
    'Lost Deals': manager_loss,
    'Percentage': manager_loss_pct
})
print(manager_summary)

LOST DEALS BY PRODUCT
                Lost Deals  Percentage
product                               
GTX Basic              521        21.1
MG Advanced            430        17.4
MG Special             430        17.4
GTXPro                 418        16.9
GTX Plus Basic         398        16.1
GTX Plus Pro           266        10.8
GTK 500                 10         0.4


LOST DEALS BY REGION
                 Lost Deals  Percentage
regional_office                        
Central                 975        39.4
West                    811        32.8
East                    687        27.8


LOST DEALS BY MANAGER
                  Lost Deals  Percentage
manager                                 
Melvin Marxen            536        21.7
Summer Sewald            459        18.6
Dustin Brinkmann         439        17.8
Rocco Neubert            422        17.1
Celia Rouche             352        14.2
Cara Losch               265        10.7
